In [1]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 
import re

In [2]:
df_adresse = pd.read_csv("../data/data_cleaned/Pseudonymisation_provisoire_geocoded_dpt_reg.csv", sep=";", dtype={"pseudo_provisoire":str})
df_adresse.drop("Unnamed: 0", axis=1)

,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,trust_score,street,...,address_has_num_init,address_has_num_geoloc,same_city,hostel,hosted,date_geoloc,geometry,CODE_IRIS,INSEE_REG,CODE_DEPT
0,1,34 RUE DES FRERES CHAUSSONS,92600,ASNIERES-SUR-SEINE,34 RUE DES FRERES CHAUSSONS 92600 ASNI...,2.289499,48.916298,0.832367,middle,34 Rue des Frères Chausson,...,True,True,True,False,False,2024-01-05,POINT (647928.3162985359 6868712.859312387),920040302,11.0,92
1,2,11 RUE EMILE DUBOIS,75014,PARIS,11 RUE EMILE DUBOIS 75014 PARIS,2.336628,48.831707,0.972567,high,11 Rue Emile Dubois,...,True,True,True,False,False,2024-01-05,POINT (651303.232633862 6859276.896689738),751145406,11.0,75
2,3,48 CHEMIN VERT,78680,EPONE,48 CHEMIN VERT 78680 EPONE,1.797376,48.950412,0.960861,high,48 Chemin Vert,...,True,True,True,False,False,2024-01-05,POINT (611921.2622170823 6872942.907671936),782170102,11.0,78
3,4,18 ALLEE DE LA CHARNILLE,47140,SAINT-SYLVESTRE-SUR-LOT,18 ALLEE DE LA CHARNILLE 47140 SAIN...,0.809474,44.404892,0.805540,middle,18 Allée de la Charmille,...,True,True,True,False,False,2024-01-05,POINT (525576.5290254143 6369736.268092031),472800000,75.0,47
4,5,31 RUE DU GENERAL DE MIRIBEL,92500,RUEIL-MALMAISON,31 RUE DU GENERAL DE MIRIBEL 92500 RUEI...,2.173326,48.865232,0.973612,high,31 Rue du Général de Miribel,...,True,True,True,False,False,2024-01-05,POINT (639354.9912859926 6863117.583096649),920630504,11.0,92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64289,64290,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD,3 AV DE FOUILLEUSE 92210 SAIN...,2.211229,48.861208,0.749249,middle,3 Avenue de Fouilleuse,...,True,True,True,False,False,2024-01-05,POINT (642131.0004121892 6862641.7109631905),920640101,11.0,92
64290,64291,81 COTE DU TORCHON,27220,L'HABIT,81 COTE DU TORCHON 27220 L'HABIT,1.365413,48.871685,0.946441,high,81 Cote du Torchon,...,True,True,True,False,False,2024-01-05,POINT (580107.5206945667 6864758.656890908),273090000,28.0,27
64291,64292,60 RUE BAUDRICOURT,75013,Paris 13,60 RUE BAUDRICOURT 75013 Pari...,2.362960,48.825882,0.811235,middle,60 Rue Baudricourt,...,True,True,True,False,False,2024-01-05,POINT (653230.9422174117 6858613.30609641),751135006,11.0,75
64292,64293,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON,159 AVENUE DE LA REPUBLIQUE 92320 CHAT...,2.300784,48.809219,0.970667,high,159 Avenue de la République,...,True,True,True,False,False,2024-01-05,POINT (648649.9241903964 6856799.224564967),920200101,11.0,92


## Combien de patients au total? 
Avec des informations

In [3]:
a = len(df_adresse)
b = df_adresse["pseudo_provisoire"].isna().sum()
c= a-b
print(a)
print(b)
print(a-b)
print(c+b)

64294
0
64294
64294


In [4]:
df_adresse_no_na = df_adresse.dropna(subset="pseudo_provisoire")

## Combien ont une adresse?

In [5]:
print(f"Nombre d'adresse nan {df_adresse_no_na["adresse"].isna().sum()}")
print(f"Nombre d'adresse null {df_adresse_no_na["adresse"].isnull().sum()}")

df_adresse_noNan = df_adresse_no_na.dropna(subset=["adresse"])
print(len(df_adresse_noNan))

Nombre d'adresse nan 1148
Nombre d'adresse null 1148
63146


## Combien ont une géométrie valide? 

In [6]:
#On supp geometry nulles => combien? MAJ flowchart 
gdf_noNan = (gpd.GeoDataFrame(df_adresse_noNan, geometry=gpd.points_from_xy(df_adresse_noNan.x, df_adresse_noNan.y)).set_crs(epsg=4326))

print(f"Nombre de géométries non valides : {len(gdf_noNan[gdf_noNan["geometry"].is_empty])}")
print(f"Nombre de géométries valides : {len(gdf_noNan[~gdf_noNan["geometry"].is_empty])}")
print(f"Verification : {len(gdf_noNan[gdf_noNan["geometry"].is_empty]) + len(gdf_noNan[~gdf_noNan["geometry"].is_empty])}")


gdf_noNan_geom = gdf_noNan[~gdf_noNan["geometry"].is_empty]


Nombre de géométries non valides : 563
Nombre de géométries valides : 62583
Verification : 63146


## Combien ont une addresse en France métropolitaine? 

In [7]:
df_dept =  gpd.read_file('H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.shp')
df_france = df_dept.dissolve()

In [8]:
gdf_noNan_geom = gdf_noNan_geom.to_crs(epsg="2154")

patients_in_france_metrop = gpd.sjoin(gdf_noNan_geom, df_france, how='inner', predicate='within')
#patients_in_france_metrop = pd.merge(gdf_noNan_geom, df_france, how="left", on="CODE_DEPT")
#patients_in_france_metrop.drop('index_right', axis=1, inplace=True)

df_patients_FR = gdf_noNan_geom[gdf_noNan_geom.index.isin(patients_in_france_metrop.index)]


# df_fr = pd.read_csv("../data/data_cleaned/patients_FR_geocoded.csv", sep=";")

#print(df_patients_FR.shape)
#print(df_fr.shape)
df_patients_FR.to_csv("../data/data_cleaned/patients_FR_geocoded.csv",sep=";")
#df_patients_FR.to_file("../data/data_cleaned/patients_FR_geocoded.shp")

In [9]:
patients_not_in_france_metrop = gdf_noNan_geom[~gdf_noNan_geom.index.isin(patients_in_france_metrop.index)]


In [10]:
print(len(patients_in_france_metrop))
# print(len(gdf_noNan_geom))
print(len(patients_not_in_france_metrop))
print(f"verif : {len(patients_in_france_metrop)+len(patients_not_in_france_metrop)}")
print(len(gdf_noNan_geom))

61911
672
verif : 62583
62583


In [62]:
patients_not_in_france_metrop[patients_not_in_france_metrop["nom_commune_postal"]=='CASE PILOTE']

['ALLEE PORTO RICO                   LOTISSEMENT LA CARAIBE             ']

In [11]:
li = ['06500 MEXICO DF MEXIQUE','KOWEIT CITY','16052-KOUBA - ALGER',
       'CASABLANCA  MAROC','BRAZZAVILLE','20100 CASABLANCA MAROC',
       'DUMBEA CEDEX','4682 HOUTAIN SAINT SIMEON','GROS-MORNE',
        'SAINTE-LUCE', 'MAMOUDZOU', 'CHIRONGUI','PRATO ITALIE', '20000 CASABLANCA MAROC',
        '8005 BERTRANGE','RABAT MAROC','2040- RADES','ALGER - ALGERIE','POINTE NOIRE - CONGO',
        '10000 HANOI VIETNAM', 'STE MARIE', 'GRAND BOIS', 'MILAN 20123 ITALIE', 'BUCAREST', 'KOWEIT', 'AMSTERDAM PAYS BAS',
       '00176 ROME ITALIE','BERGAMO','ANTANANARIVO MADAGASCAR']
liste= ['MORNE A L EAU', 'FORT DE FRANCE', 'BAIE-MAHAULT', 'DEAUVILLE',
       'SAINT-MARTIN','ANSE-BERTRAND', 'LA MONTAGNE', 'BOUILLANTE', 'GRAND BOURG',
       'GOYAVE', 'SAINT-LAURENT-DU-MARONI', 'SAINTE-ROSE', 'PETIT-BOURG',
       'LE ROBERT', 'RAVINE DES CABRIS', 'SAINT-PIERRE', 'STE ANNE',
       "L'ETANG-SALE", 'LAMENTIN', 'LE GOSIER', 'WALLIS', 'LE LAMENTIN',
       'CAYENNE', 'ABYMES', 'LE VAUCLIN', 'LES ABYMES',
       'ST DENIS DE LA REUNION', 'REMIRE-MONTJOLY','SCHOELCHER','SAINTE-MARIE', 'SAINTE-ANNE', 'LA MONTAGNE.LA REUNION',
       'PETIT-CANAL', 'LE DIAMANT', 'LE MOULE','FAKARAVA','POINTE-NOIRE', 'SAINT-BARTHELEMY', 'BOIS DE NEFLES ST PAUL',
       'BASSE-POINTE', 'SAINT-CLAUDE', 'PORT-LOUIS', 'MATOURY CAYENNE',
       'ST ESPRIT', 'CHERBOURG', 'GROS MORNE', "MORNE A L' EAU",
       'LA POSSESSION', 'PETIT BOURG GUADELOUPE', 'REMIRE MONTJOLY',
       'ST LAURENT DU MARONI', 'LE PITON SAINT LEU', 'LA TRINITE','FAAA','NOUMEA CEDEX',
       'SAINT-FRANCOIS', 'MATOURY',
       'MOTU UTA - TAHITI - POLYNESIE FRANCAISE', 'COMBANI',
       'CAPESTERRE BELLE EAU', 'SAINT DENIS', 'DUCOS', 'SAINTE MARIE',
       'CAPESTERRE-BELLE-EAU', 'SAINT-JOSEPH', 'ST DENIS',
       'VIEUX-HABITANTS', 'MARIPASOULA', 'LE FRANCOIS', 'BAIE MAHAULT',
       'ST ANDRE', 'BASSE TERRE', 'SAINT LEU', 'LABATTOIR - MAYOTTE',
       'LE CARBET', 'MACOURIA', 'PUNAAUIA TAHITI', "MORNE-A-L'EAU",
       'TROIS-RIVIERES', 'PETITE-ILE', 'KOUROU', 'ST PIERRE', 'MANA',
       'REGINA', 'RIVIERE PILOTE','DZAOUDZI LABATTOIR','TAHITI POLYNESIE FRANCAISE', 'LE LORRAIN', 'LES TROIS ILETS',
       'SAINT DENIS CEDEX', 'LA CHALOUPE', 'PETIT CANAL',
       'SAINT-DENIS', 'TSINGONI',
       'ST PIERRE / LA REUNION', 'LA PLAINE-DES-PALMISTES','NOUMEA NOUVELLE CALEDONIE', 'CAMOPI', 'ST PAUL', 'RIVIERE-SALEE',
       'MASSY', 'LES AVIRONS', 'FORT-DE-FRANCE','POINTE A PITRE', 'NOUMEA NELLE CALEDONIE', 'MONT DORE', 'BAILLIF',
       'BOISRIPEAUX ABIMES', 'ST MARTIN', 'DESHAIES', 'PIRAE',
       'LE TAMPON', 'ROYAN', 'St Leu', 'ST BARTHELEMY',
       'ST BENOIT', 'PETIT BOURG','KOUNDOU - MAYOTTE',
       'LA RIVIERE SAINT LOUIS', 'MACOUBA', 'SAINTE CLOTILDE', 'ST LOUIS',
       'SAINT-ANDRE', 'SAINT ANDRE', 'LE MARIGOT-MARTINIQUE',
       'MONT DORE NOUVELLE CALEDONIE', 'POINTE-A-PITRE',
       'TERRE-DE-HAUT (GUADELOUPE)', 'SAINT-ESPRIT', 'SAINT-PAUL',
       'LES TROIS-ILETS', 'ST DENIS CEDEX', 'LE PITON ST LEU',
       'ST JOSEPH', 'CASE-PILOTE', 'LE BLANC-MESNIL', 'ST CLAUDE','NOUMEA', 'RIVIERE SALEE', 'MAMOUDZOU - MAYOTTE', 'STE CLOTILDE',
       'VIEUX HABITANTS', 'RIVIERE-PILOTE', 'SCHOELCHER MARTINIQUE','NABEUL', 'SAINT-PIERRE-ET-MIQUELON','CASE PILOTE', 'ST PIERRE ET MIQUELON', 'PAPEETE TAHITI',
       'LA SALINE','MONTSINERY TONNEGRANDE','BASSE POINTE', 'ISSY-LES-MOULINEAUX', 'ST GILLES LES BAINS','PANLATTE','LA DESIRADE',
       'STE LUCE','L ETANG SALE','ST FRANCOIS','MACOURIA TONATE', 'STE SUZANNE','LE QUESNOY','LE PORT','THEOULE SUR MER','CANNES',
       'RAVENOVILLE PLAGE','GROSSETO PRUGNA','BELZ','PORTICCIO',"L'ILE-D'YEU"]

df_fr_metrop = patients_not_in_france_metrop[(patients_not_in_france_metrop["nom_commune_postal"].isin(liste))]
df_etrangers = patients_not_in_france_metrop[patients_not_in_france_metrop["nom_commune_postal"].isin(li)]
df_autre = patients_not_in_france_metrop[(~patients_not_in_france_metrop["nom_commune_postal"].isin(li))&(~patients_not_in_france_metrop["nom_commune_postal"].isin(liste))]

In [12]:
df_fr_metrop

,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,trust_score,...,address_has_num_init,address_has_num_geoloc,same_city,hostel,hosted,date_geoloc,geometry,CODE_IRIS,INSEE_REG,CODE_DEPT
14,14,15,BOSREDON,97111,MORNE A L EAU,BOSREDON 97111 MORN...,-61.488462,16.311416,0.972554,high,...,False,False,True,False,False,2024-01-05,POINT (-6258235.351 6119869.331),NaN,NaN,NaN
148,148,149,36 ROCADE DU BEL HORIZON REDOUTE,97200,FORT DE FRANCE,36 ROCADE DU BEL HORIZON REDOUTE 97200 FORT...,-61.055868,14.644075,0.758602,middle,...,True,True,True,False,False,2024-01-05,POINT (-6374710.558 5937134.783),NaN,NaN,NaN
457,457,458,3 IMPASSE L AJOUPA LA RETRAITE...,97122,BAIE-MAHAULT,3 IMPASSE L AJOUPA LA RETRAITE...,-61.615246,16.228373,0.595055,middle,...,True,False,True,False,False,2024-01-05,POINT (-6276311.917 6123942.318),NaN,NaN,NaN
467,467,468,4 QUAI DES MARCHANDS,14800,DEAUVILLE,4 QUAI DES MARCHANDS 14800 DEAU...,0.072231,49.366103,0.952180,high,...,True,True,True,False,False,2024-01-05,POINT (487328.502 6922451.429),142200000,28.0,NaN
501,501,502,VOIE 34 LA FERME REDOUTE 11 IMPASSE ...,97200,FORT DE FRANCE,VOIE 34 LA FERME REDOUTE 11 IMPASSE ...,-61.051675,14.638450,0.533150,middle,...,False,True,True,False,False,2024-01-05,POINT (-6374871.408 5936267.510),NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63659,63659,63660,701RESIDENCE BOIS VERT SAINT PHY ...,97120,ST CLAUDE,701RESIDENCE BOIS VERT SAINT PHY ...,-61.717006,16.004189,0.440035,middle,...,True,False,True,False,False,2024-01-05,POINT (-6305267.829 6113752.984),NaN,NaN,NaN
63759,63759,63760,31 AVENUE DU PETIT PARADIS,97233,SCHOELCHER,31 AVENUE DU PETIT PARADIS 97233 SCHO...,-61.086078,14.619159,0.967206,high,...,True,True,True,False,False,2024-01-05,POINT (-6379574.373 5937665.209),NaN,NaN,NaN
63793,63793,63794,QUARTIER DUFRESNE,97215,RIVIERE SALEE,QUARTIER DUFRESNE 97215 RIVI...,-60.934236,14.522135,0.960325,high,...,False,False,True,False,False,2024-01-05,POINT (-6375563.844 5915565.970),NaN,NaN,NaN
63965,63965,63966,PARNASSE RUE DES COMMIERS BLANCS,97120,SAINT-CLAUDE,PARNASSE RUE DES COMMIERS BLANCS 97120 SAIN...,-61.687357,16.029179,0.668136,middle,...,False,False,True,False,False,2024-01-05,POINT (-6300519.874 6113264.252),NaN,NaN,NaN


In [48]:
df_fr_CO = df_fr_metrop[df_fr_metrop["codepost"].str.startswith(('971',"972","988","974","977","978","986","987","973","976","980","975"))]
df_fr_err = df_fr_metrop[~df_fr_metrop["codepost"].str.startswith(('971',"972","988","974","977","978","986","987","973","976","980","975"))]

In [ ]:
print(f"Nombre de patients hors france metrop mais fr {len(df_fr_err)}")
print(f"Nombre de patients hors france metrop et pas fr {len(df_etrangers)}")
print(f"Nombre de patients hors france metrop mais CO {len(df_fr_CO)}")
print(f"verif : {len(df_fr_err)+len(df_etrangers)+len(df_fr_CO)} . {len(patients_not_in_france_metrop)}")